# Tutorial 3: Protein Embeddings

This notebook demonstrates how to compute **protein-level embeddings** with
`embpy`, using the ESM family of protein language models.

Available protein embedding models:

| Model key | Parameters | Architecture |
|---|---|---|
| `esm2_8M` | 8 M | ESM-2 |
| `esm2_35M` | 35 M | ESM-2 |
| `esm2_150M` | 150 M | ESM-2 |
| `esm2_650M` | 650 M | ESM-2 |
| `esm2_3B` | 3 B | ESM-2 |
| `esmc_300m` | 300 M | ESM-C |
| `esmc_600m` | 600 M | ESM-C |

These models take amino-acid sequences as input. The `BioEmbedder` will
automatically fetch protein sequences from UniProt when you provide a
gene symbol or Ensembl ID.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

## 1. Embed a single gene as a protein

The `embed_gene` method detects that you are using a protein model and
will fetch the corresponding amino-acid sequence automatically.

In [ ]:
MODEL = "esm2_650M"  # good balance of quality and speed

emb = embedder.embed_gene(
    identifier="TP53",
    model=MODEL,
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
)

print(f"Shape:  {emb.shape}")
print(f"Dtype:  {emb.dtype}")
print(f"Norm:   {(emb ** 2).sum() ** 0.5:.2f}")

## 2. Provide a raw protein sequence

If you already have the amino-acid sequence, pass it directly. The
identifier detector will recognize it as a protein sequence.

In [ ]:
# First 100 residues of human p53 (UniProt P04637)
p53_frag = (
    "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP"
    "DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQK"
)

emb_seq = embedder.embed_gene(
    identifier=p53_frag,
    model=MODEL,
    pooling_strategy="mean",
)
print(f"Fragment embedding shape: {emb_seq.shape}")

## 3. Batch protein embedding

In [ ]:
genes = ["TP53", "BRCA1", "EGFR", "KRAS", "MYC"]

embeddings = embedder.embed_genes_batch(
    model=MODEL,
    identifiers=genes,
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
)

for gene, emb in zip(genes, embeddings):
    if emb is not None:
        print(f"  {gene:8s} → dim {emb.shape[0]:,}")
    else:
        print(f"  {gene:8s} → FAILED")

## 4. Comparing ESM-2 model sizes

Larger ESM-2 checkpoints produce higher-quality but higher-dimensional
embeddings. Here we compare the output dimensionality.

In [ ]:
for m in ["esm2_8M", "esm2_35M", "esm2_150M", "esm2_650M"]:
    try:
        e = embedder.embed_gene("TP53", model=m, pooling_strategy="mean")
        print(f"  {m:15s} → dim {e.shape[0]:,}")
    except Exception as exc:
        print(f"  {m:15s} → {exc}")

## 5. Pooling strategies for protein models

Protein language models output one vector per residue. Pooling reduces
these to a single vector:

- `"mean"` – average across residues (most common)
- `"max"` – max-pool across residues
- `"cls"` – use the CLS token embedding

In [ ]:
import numpy as np

for strategy in ["mean", "max", "cls"]:
    try:
        e = embedder.embed_gene("TP53", model=MODEL, pooling_strategy=strategy)
        print(f"  {strategy:5s} → shape {e.shape}, L2 norm = {np.linalg.norm(e):.2f}")
    except Exception as exc:
        print(f"  {strategy:5s} → {exc}")

## 6. Fetching protein sequences separately

You can also use the `GeneResolver` to fetch protein sequences first,
inspect or filter them, and then embed.

In [ ]:
from embpy.resources.gene_resolver import GeneResolver

resolver = GeneResolver(auto_download=False)

seq = resolver.get_protein_sequence("BRCA1", id_type="symbol")
if seq:
    print(f"BRCA1 protein length: {len(seq)} aa")
    print(f"First 80 aa: {seq[:80]}…")

In [ ]:
# Batch fetch protein sequences
prot_seqs = resolver.get_protein_sequences_batch(
    ["TP53", "BRCA1", "EGFR"],
    id_type="symbol",
)
for gene_id, seq in prot_seqs.items():
    print(f"  {gene_id:8s} → {len(seq):,} aa")

## 7. Using ESM-C models

ESM-C is a more recent architecture from EvolutionaryScale.
Usage is identical.

In [ ]:
try:
    emb_esmc = embedder.embed_gene(
        "TP53", model="esmc_300m", pooling_strategy="mean",
    )
    print(f"ESM-C 300M embedding: shape {emb_esmc.shape}")
except Exception as exc:
    print(f"ESM-C not available: {exc}")

## Summary

| What | How |
|---|---|
| Protein embedding by gene symbol | `embed_gene("TP53", model="esm2_650M")` |
| Protein embedding by raw sequence | `embed_gene("MTEYKLV…", model="esm2_650M")` |
| Batch | `embed_genes_batch(model, identifiers=[…])` |
| Fetch sequences separately | `GeneResolver.get_protein_sequence()` |

Next: [Tutorial 4 – Molecule Embeddings](04_molecule_embeddings.ipynb)